In [0]:
%pip install soda-core-spark-df

In [0]:
# =============================================================================
# F1-Pulse | Data Quality — Gold Layer
# Notebook: 05_Check_Gold
# Runs SODA checks against Gold tables.
# Raises exception on failure to halt the Databricks job.
# =============================================================================

import os
from soda.scan import Scan
from config.config import CATALOG, GOLD, MIN_LAP_DURATION_S

# ---------------------------------------------------------------------------
# 1. Load Gold tables from metastore
# ---------------------------------------------------------------------------
df_driver_metrics = spark.table(f"{CATALOG}.{GOLD}.driver_performance_metrics")
df_constructors   = spark.table(f"{CATALOG}.{GOLD}.constructor_standings")
df_progression    = spark.table(f"{CATALOG}.{GOLD}.lap_progression")

# ---------------------------------------------------------------------------
# 2. Register temp views (underscore names — dots not allowed in view names)
# ---------------------------------------------------------------------------
df_driver_metrics.createOrReplaceTempView("gold_driver_performance_metrics")
df_constructors.createOrReplaceTempView("gold_constructor_standings")
df_progression.createOrReplaceTempView("gold_lap_progression")

# ---------------------------------------------------------------------------
# 3. Load and patch YAML
#    - Patch table names to match temp view names
#    - Patch lap duration threshold from central config (single source of truth)
# ---------------------------------------------------------------------------
repo_root = "/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse"
with open(f"{repo_root}/soda/checks_gold.yml", "r") as f:
    yaml_content = f.read()

# Table name patches
yaml_content = yaml_content.replace(
    "checks for gold.driver_performance_metrics",
    "checks for gold_driver_performance_metrics"
)
yaml_content = yaml_content.replace(
    "checks for gold.constructor_standings",
    "checks for gold_constructor_standings"
)
yaml_content = yaml_content.replace(
    "checks for gold.lap_progression",
    "checks for gold_lap_progression"
)

# Threshold patch — keeps YAML in sync with config.py automatically
yaml_content = yaml_content.replace(">= 60", f">= {MIN_LAP_DURATION_S}")

# ---------------------------------------------------------------------------
# 4. Run Soda scan
# ---------------------------------------------------------------------------
scan = Scan()
scan.set_data_source_name("spark_df")
scan.add_spark_session(spark)
scan.add_sodacl_yaml_str(yaml_content)

exit_code = scan.execute()
print(scan.get_logs_text())

# ---------------------------------------------------------------------------
# 5. Halt pipeline on failure — Databricks job will mark task as FAILED
# ---------------------------------------------------------------------------
if exit_code != 0:
    raise Exception("❌ Gold DQ checks failed. Halting pipeline.")

print("✅ Gold DQ checks passed.")